# Exploratory Data Analysis — New Acquisitions CLV Model

Preliminary exploration of the preprocessed DataFrame (`df_acquisitions.pkl`) before running formal hypothesis tests.

**Sections:**
1. Load & Overview
2. Target Distribution (`pnb_total`)
3. Window Variables (`n_mois_nbi`, `n_mois_observation`)
4. Segments & Professions
5. Equipment Overview
6. Early NBI Behavior
7. Correlation Matrix
8. Cohort Overview
9. Key Takeaways

In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

plt.rcParams.update({
    "figure.figsize": (14, 6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

sns.set_palette("muted")

df = pickle.load(open("df_acquisitions.pkl", "rb"))
print(f"DataFrame: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 1. Overview

In [ ]:
print("--- Column types ---")
print(df.dtypes.value_counts().to_string())

print("\n--- First rows ---")
df.head(3)

In [ ]:
print("--- Numeric summary ---")
df.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T.round(1)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_report = pd.DataFrame({"n_missing": missing, "pct": missing_pct})
missing_report = missing_report[missing_report["n_missing"] > 0].sort_values("pct", ascending=False)

if len(missing_report) > 0:
    print(f"Columns with missing values: {len(missing_report)}")
    display(missing_report)
    
    fig, ax = plt.subplots(figsize=(12, max(4, len(missing_report) * 0.3)))
    missing_report["pct"].sort_values().plot(kind="barh", ax=ax, color="salmon")
    ax.set_xlabel("% missing")
    ax.set_title("Missing values by column")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values detected.")

## 2. Target Distribution: `pnb_total`

In [ ]:
pnb = df["pnb_total"].dropna()

print(f"Count:    {len(pnb):,}")
print(f"Mean:     {pnb.mean():,.0f} EUR")
print(f"Median:   {pnb.median():,.0f} EUR")
print(f"Std:      {pnb.std():,.0f} EUR")
print(f"Min:      {pnb.min():,.0f} EUR")
print(f"Max:      {pnb.max():,.0f} EUR")
print(f"Skew:     {pnb.skew():.2f}")
print(f"Kurtosis: {pnb.kurtosis():.2f}")
print(f"PNB <= 0: {(pnb <= 0).sum()} ({(pnb <= 0).mean()*100:.1f}%)")
print(f"PNB = 0:  {(pnb == 0).sum()} ({(pnb == 0).mean()*100:.1f}%)")

q1, q3 = pnb.quantile(0.25), pnb.quantile(0.75)
iqr = q3 - q1
outliers_3iqr = ((pnb < q1 - 3*iqr) | (pnb > q3 + 3*iqr)).sum()
print(f"\nOutliers (3x IQR): {outliers_3iqr} ({outliers_3iqr/len(pnb)*100:.1f}%)")
print(f"IQR bounds: [{q1 - 3*iqr:,.0f}, {q3 + 3*iqr:,.0f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(pnb.clip(-5000, 20000), bins=100, edgecolor="white", alpha=0.8)
axes[0].axvline(pnb.mean(), color="red", ls="--", label=f"Mean={pnb.mean():,.0f}")
axes[0].axvline(pnb.median(), color="green", ls="--", label=f"Median={pnb.median():,.0f}")
axes[0].set_xlabel("PNB total (EUR)")
axes[0].set_title("Distribution of pnb_total (clipped)")
axes[0].legend()

axes[1].hist(np.log1p(pnb.clip(lower=0)), bins=80, edgecolor="white", alpha=0.8, color="green")
axes[1].set_xlabel("log(PNB + 1)")
axes[1].set_title("Log-transformed (PNB > 0)")

axes[2].boxplot([pnb.values], vert=True, showfliers=False, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6))
axes[2].set_ylabel("PNB total (EUR)")
axes[2].set_title("Boxplot (no outliers)")

plt.suptitle("Target Variable: pnb_total", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
if "pnb_remaining" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(df["pnb_remaining"].dropna().clip(-5000, 15000), bins=80, edgecolor="white", alpha=0.8, color="coral")
    axes[0].set_xlabel("PNB remaining (EUR)")
    axes[0].set_title("Distribution of pnb_remaining")

    axes[1].scatter(df["pnb_cumul_obs"], df["pnb_remaining"], alpha=0.1, s=5)
    axes[1].set_xlabel("pnb_cumul_obs")
    axes[1].set_ylabel("pnb_remaining")
    axes[1].set_title("Observed vs Remaining NBI")
    plt.tight_layout()
    plt.show()

## 3. Window Variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [
    (axes[0], "n_mois_nbi", "n_mois_nbi (target duration)"),
    (axes[1], "n_mois_observation", "n_mois_observation (feature window)"),
]:
    if col in df.columns:
        counts = df[col].value_counts().sort_index()
        counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
        ax.set_title(title)
        ax.set_ylabel("Count")
        for i, (idx, v) in enumerate(counts.items()):
            ax.text(i, v + len(df)*0.005, f"{v/len(df)*100:.1f}%", ha="center", fontsize=10)
        plt.sca(ax)
        plt.xticks(rotation=0)
    else:
        ax.set_title(f"{col} — NOT FOUND")

plt.tight_layout()
plt.show()

if "n_mois_nbi" in df.columns and "n_mois_observation" in df.columns:
    print("Cross-tab n_mois_nbi x n_mois_observation:")
    display(pd.crosstab(df["n_mois_nbi"], df["n_mois_observation"]))

In [ ]:
if "pnb_mensuel_moyen" in df.columns and "n_mois_nbi" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    df.boxplot(column="pnb_total", by="n_mois_nbi", ax=axes[0], showfliers=False, patch_artist=True)
    axes[0].set_title("pnb_total by n_mois_nbi")
    axes[0].set_xlabel("n_mois_nbi")
    axes[0].set_ylabel("EUR")

    df.boxplot(column="pnb_mensuel_moyen", by="n_mois_nbi", ax=axes[1], showfliers=False, patch_artist=True)
    axes[1].set_title("pnb_mensuel_moyen by n_mois_nbi")
    axes[1].set_xlabel("n_mois_nbi")
    axes[1].set_ylabel("EUR / month")

    plt.suptitle("")
    plt.tight_layout()
    plt.show()

## 4. Segments & Professions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if "segment" in df.columns:
    seg_counts = df["segment"].value_counts()
    seg_counts.plot(kind="bar", ax=axes[0], color="teal", edgecolor="white")
    axes[0].set_title("Client count by segment")
    axes[0].set_ylabel("Count")
    for i, (idx, v) in enumerate(seg_counts.items()):
        axes[0].text(i, v + len(df)*0.005, f"{v/len(df)*100:.1f}%", ha="center", fontsize=10)
    plt.sca(axes[0])
    plt.xticks(rotation=15, ha="right")

    seg_pnb = df.groupby("segment")["pnb_total"].mean().sort_values(ascending=False)
    seg_pnb.plot(kind="bar", ax=axes[1], color=["#e74c3c", "#3498db", "#2ecc71", "#f39c12"][:len(seg_pnb)], edgecolor="white")
    axes[1].set_title("Mean pnb_total by segment")
    axes[1].set_ylabel("EUR")
    plt.sca(axes[1])
    plt.xticks(rotation=15, ha="right")

plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
if "profession" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    prof_counts = df["profession"].value_counts()
    prof_counts.plot(kind="barh", ax=axes[0], color="steelblue", edgecolor="white")
    axes[0].set_title("Client count by profession")
    axes[0].set_xlabel("Count")

    prof_pnb = df.groupby("profession")["pnb_total"].mean().sort_values()
    prof_pnb.plot(kind="barh", ax=axes[1], color="teal", edgecolor="white")
    axes[1].set_title("Mean pnb_total by profession")
    axes[1].set_xlabel("EUR")

    plt.tight_layout()
    plt.show()

In [ ]:
if "segment" in df.columns and "profession" in df.columns:
    pivot = df.pivot_table(values="pnb_total", index="profession", columns="segment", aggfunc="mean").round(0)
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(pivot, annot=True, fmt=",.0f", cmap="YlOrRd", ax=ax, linewidths=0.5)
    ax.set_title("Mean PNB total — Segment x Profession")
    plt.tight_layout()
    plt.show()

## 5. Equipment Overview

In [ ]:
qty_cols = [c for c in df.columns if c.startswith("qty_")]
print(f"Equipment columns (qty_*): {len(qty_cols)}")

if qty_cols:
    equip_summary = pd.DataFrame({
        "non_null": df[qty_cols].notna().sum(),
        "non_zero": (df[qty_cols].fillna(0) > 0).sum(),
        "pct_holders": ((df[qty_cols].fillna(0) > 0).sum() / len(df) * 100).round(1),
        "mean_qty": df[qty_cols].mean().round(2),
    }).sort_values("pct_holders", ascending=False)
    display(equip_summary)

if "n_produits_total" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df["n_produits_total"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
    axes[0].set_title("Distribution of n_produits_total")
    axes[0].set_xlabel("Number of products")
    axes[0].set_ylabel("Count")
    plt.sca(axes[0])
    plt.xticks(rotation=0)

    prod_pnb = df.groupby("n_produits_total")["pnb_total"].mean()
    prod_pnb.plot(kind="bar", ax=axes[1], color="teal", edgecolor="white")
    axes[1].set_title("Mean pnb_total by n_produits_total")
    axes[1].set_xlabel("Number of products")
    axes[1].set_ylabel("EUR")
    plt.sca(axes[1])
    plt.xticks(rotation=0)

    plt.tight_layout()
    plt.show()

In [ ]:
if qty_cols:
    impact = []
    for col in qty_cols:
        has = df[df[col].fillna(0) > 0]["pnb_total"]
        hasnt = df[df[col].fillna(0) == 0]["pnb_total"]
        if len(has) > 10 and len(hasnt) > 10:
            impact.append({
                "product": col,
                "n_with": len(has),
                "pnb_with": has.mean(),
                "pnb_without": hasnt.mean(),
                "diff": has.mean() - hasnt.mean(),
            })
    impact_df = pd.DataFrame(impact).sort_values("diff", ascending=False)

    fig, ax = plt.subplots(figsize=(12, max(4, len(impact_df) * 0.35)))
    colors = ["#2ecc71" if d > 0 else "#e74c3c" for d in impact_df["diff"].values]
    ax.barh(impact_df["product"], impact_df["diff"], color=colors)
    ax.set_xlabel("Difference in mean PNB (with - without product)")
    ax.set_title("Product impact on pnb_total")
    ax.axvline(0, color="black", lw=0.5)
    plt.tight_layout()
    plt.show()

## 6. Early NBI Behavior

In [ ]:
pnb_monthly = sorted([c for c in df.columns if c.startswith("pnb_m") and c[4:].isdigit()],
                      key=lambda x: int(x.split("_m")[1]))
print(f"Monthly PNB columns: {pnb_monthly}")

if pnb_monthly:
    monthly_means = df[pnb_monthly].mean()
    monthly_medians = df[pnb_monthly].median()
    monthly_nonnull = df[pnb_monthly].notna().sum()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(range(1, len(pnb_monthly)+1), monthly_means, "o-", color="steelblue", label="Mean")
    axes[0].plot(range(1, len(pnb_monthly)+1), monthly_medians, "s--", color="green", label="Median")
    axes[0].set_xlabel("Month since acquisition")
    axes[0].set_ylabel("PNB (EUR)")
    axes[0].set_title("Monthly PNB: mean vs median")
    axes[0].legend()

    axes[1].bar(range(1, len(pnb_monthly)+1), monthly_nonnull, color="teal", edgecolor="white")
    axes[1].set_xlabel("Month")
    axes[1].set_ylabel("Non-null count")
    axes[1].set_title("Data availability by month")
    axes[1].axhline(len(df), color="red", ls="--", alpha=0.5, label="Total clients")
    axes[1].legend()

    if "pnb_cumul_obs" in df.columns:
        axes[2].scatter(df["pnb_cumul_obs"], df["pnb_total"], alpha=0.1, s=5)
        r, p = stats.pearsonr(df["pnb_cumul_obs"].dropna(), df.loc[df["pnb_cumul_obs"].notna(), "pnb_total"])
        axes[2].set_xlabel("pnb_cumul_obs (EUR)")
        axes[2].set_ylabel("pnb_total (EUR)")
        axes[2].set_title(f"Observed vs Total NBI (r={r:.3f})")

    plt.tight_layout()
    plt.show()

In [ ]:
if pnb_monthly and len(pnb_monthly) >= 3:
    correlations = []
    for n in range(1, min(len(pnb_monthly)+1, 13)):
        cumul_n = df[pnb_monthly[:n]].sum(axis=1)
        mask = cumul_n.notna() & df["pnb_total"].notna()
        if mask.sum() > 10:
            r, _ = stats.pearsonr(cumul_n[mask], df.loc[mask, "pnb_total"])
            correlations.append({"months": n, "pearson_r": round(r, 4)})

    corr_df = pd.DataFrame(correlations)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(corr_df["months"], corr_df["pearson_r"], "o-", lw=2, markersize=8, color="steelblue")
    ax.set_xlabel("Cumulated months of NBI")
    ax.set_ylabel("Pearson r with pnb_total")
    ax.set_title("Progressive correlation: cumulated NBI vs pnb_total")
    ax.axhline(0.7, color="green", ls="--", alpha=0.5, label="Good (r=0.7)")
    ax.axhline(0.5, color="orange", ls="--", alpha=0.5, label="Moderate (r=0.5)")
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()

    display(corr_df)

In [ ]:
if "obs_slope" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(df["obs_slope"].dropna().clip(-500, 500), bins=80, edgecolor="white", alpha=0.8)
    axes[0].axvline(0, color="red", ls="--", alpha=0.5)
    axes[0].set_xlabel("obs_slope")
    axes[0].set_title("Distribution of NBI slope (observation window)")

    if "obs_volatility" in df.columns:
        axes[1].hist(df["obs_volatility"].dropna().clip(0, 3000), bins=80, edgecolor="white", alpha=0.8, color="coral")
        axes[1].set_xlabel("obs_volatility")
        axes[1].set_title("Distribution of NBI volatility")

    plt.tight_layout()
    plt.show()

## 7. Correlation Matrix

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

key_cols = [c for c in [
    "pnb_total", "pnb_cumul_obs", "pnb_cumul_3m", "pnb_remaining", "pnb_mensuel_moyen",
    "n_mois_nbi", "n_mois_observation", "obs_slope", "obs_volatility", "obs_active_months",
    "n_produits_total",
] if c in numeric_cols]

if len(key_cols) >= 3:
    corr = df[key_cols].corr().round(3)

    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                ax=ax, linewidths=0.5, vmin=-1, vmax=1, square=True)
    ax.set_title("Correlation matrix — key features")
    plt.tight_layout()
    plt.show()

In [ ]:
if len(numeric_cols) > 5:
    full_corr = df[numeric_cols].corr()
    target_corr = full_corr["pnb_total"].drop("pnb_total", errors="ignore").sort_values(ascending=False)

    print("Top 15 features correlated with pnb_total:")
    print(target_corr.head(15).to_string())
    print("\nBottom 5:")
    print(target_corr.tail(5).to_string())

    high_corr_pairs = []
    for i in range(len(full_corr.columns)):
        for j in range(i+1, len(full_corr.columns)):
            r = full_corr.iloc[i, j]
            if abs(r) > 0.8:
                high_corr_pairs.append((full_corr.columns[i], full_corr.columns[j], round(r, 3)))
    if high_corr_pairs:
        print(f"\nHighly correlated pairs (|r| > 0.8): {len(high_corr_pairs)}")
        for a, b, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2]))[:15]:
            print(f"  {a} <-> {b}: r={r}")
    else:
        print("\nNo highly correlated pairs (|r| > 0.8) found.")

## 8. Cohort Overview

In [ ]:
if "annee_acquisition" in df.columns and "trimestre_acquisition" in df.columns:
    cohort_stats = df.groupby(["annee_acquisition", "trimestre_acquisition"]).agg(
        n_clients=("client_id", "count"),
        pnb_mean=("pnb_total", "mean"),
        pnb_median=("pnb_total", "median"),
    ).round(0)

    display(cohort_stats)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    labels = [f"{y}-{q}" for y, q in cohort_stats.index]
    axes[0].bar(labels, cohort_stats["n_clients"], color="steelblue", edgecolor="white")
    axes[0].set_title("Acquisitions per cohort")
    axes[0].set_ylabel("Count")
    plt.sca(axes[0])
    plt.xticks(rotation=45, ha="right")

    axes[1].bar(labels, cohort_stats["pnb_mean"], color="teal", edgecolor="white")
    axes[1].set_title("Mean pnb_total per cohort")
    axes[1].set_ylabel("EUR")
    plt.sca(axes[1])
    plt.xticks(rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

In [ ]:
if "annee_acquisition" in df.columns and "segment" in df.columns:
    seg_mix = pd.crosstab(df["annee_acquisition"], df["segment"], normalize="index") * 100
    print("Segment mix by year (%):\n")
    display(seg_mix.round(1))

    fig, ax = plt.subplots(figsize=(10, 6))
    seg_mix.plot(kind="bar", stacked=True, ax=ax, edgecolor="white")
    ax.set_title("Segment mix evolution by acquisition year")
    ax.set_ylabel("%")
    ax.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
pnb_positive = df[df["pnb_total"] > 0]["pnb_total"].sort_values(ascending=False)
n = len(pnb_positive)
total = pnb_positive.sum()

print("Concentration of pnb_total (Pareto analysis):")
for pct in [1, 5, 10, 20, 50]:
    top_n = max(int(n * pct / 100), 1)
    top_sum = pnb_positive.head(top_n).sum()
    print(f"  Top {pct:2d}% ({top_n:,} clients): {top_sum/total*100:.1f}% of total PNB")

cum_pnb = np.cumsum(pnb_positive.values) / total * 100
cum_clients = np.arange(1, n+1) / n * 100

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(cum_clients, cum_pnb, lw=2, label="Lorenz curve")
ax.plot([0, 100], [0, 100], "--", color="gray", alpha=0.5, label="Perfect equality")
ax.set_xlabel("% of clients (by PNB desc)")
ax.set_ylabel("% of total PNB")
ax.set_title("Lorenz Curve — PNB Concentration")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Key Takeaways

**Fill in after running the notebook:**

| Metric | Value |
|--------|-------|
| Total clients | |
| Mean pnb_total | |
| Median pnb_total | |
| % PNB <= 0 | |
| r(pnb_cumul_obs, pnb_total) | |
| r(pnb_cumul_obs, pnb_remaining) | |
| Top 20% concentrate | % of PNB |
| Dominant segment | |
| n_mois_nbi distribution | 10 / 11 / 12 |
| Number of qty_* features | |